# SeguroData Bogotá — Notebook 02: Análisis Exploratorio de Datos (EDA)

> **Proyecto:** SeguroData Bogotá — Concurso Datos al Ecosistema 2026 — MinTIC · Reto #2 Seguridad Ciudadana  
> **Autor:** Ángel Estrada  
> **Fecha:** Mayo 2026  

Este notebook realiza el análisis exploratorio de datos sobre la capa **Silver** del sistema SeguroData.  
Los datos vienen del pipeline de transformación (`src/transform.py`) que procesa las **10 fuentes Bronze** (F1-F10).

---

## Fuentes de datos usadas en este EDA

| Archivo | Descripción | Filas | Cobertura |
|---------|-------------|------:|----------|
| `silver_upz_mes.parquet` | **Tabla principal: 20 columnas por UPZ × mes × tipo_crimen** | **111,606** | 2025–2026 |
| `delitos_localidad_anio.parquet` | DAI por localidad × año × tipo (referencia histórica) | ~2,079 | 2018–2026 |
| `f2_upz.geojson` | 112 polígonos UPZ Bogotá (base cartográfica) | 112 | Estático |
| `f3_clima_bogota.parquet` | Temperatura y precipitación diaria Bogotá | ~2,338 | 2020–2026 |
| `f6_hurto_personas.parquet` | Hurto a personas PN — nivel municipal | ~638K | 2003–2026 |

**Nota sobre las fuentes:**  
- F5 NUSE (2025–2026) genera las 111,606 filas de la Silver table. F1 DAI solo tiene 21 filas a nivel localidad × año.
- La tendencia histórica 2018–2026 se construye con F1 (DAI por localidad) + F6 (Hurto PN).  
- F1 **no tiene desglose UPZ** — la granularidad UPZ del modelo viene de F5 NUSE.
- Split temporal del modelo: TRAIN = ene–oct 2025, TEST = nov 2025–abr 2026.

---
## 0. Setup — Importaciones y rutas

In [ ]:
# ─── Instalación en Colab (descomentar si es necesario) ───────────────────────
# !pip install polars geopandas pyogrio pyarrow folium seaborn -q

import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Asegurar que el root del proyecto esté en sys.path
PROJECT_ROOT = Path('.').resolve()
if 'SeguroData' not in PROJECT_ROOT.name and PROJECT_ROOT.name != 'segurodata':
    # Si se ejecuta desde subdirectorio
    for p in PROJECT_ROOT.parents:
        if (p / 'src' / 'pipeline.py').exists():
            PROJECT_ROOT = p
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR  = PROJECT_ROOT / 'datos' / 'raw'
PROC_DIR = PROJECT_ROOT / 'datos' / 'procesados'
GRAF_DIR = PROJECT_ROOT / 'graficas'
GRAF_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Archivos procesados: {list(PROC_DIR.glob("*.parquet")) + list(PROC_DIR.glob("*.csv"))}')

In [ ]:
import polars as pl
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import folium
from folium.plugins import HeatMap

# Estilo global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print('Librerías cargadas ✓')
print(f'  polars   {pl.__version__}')
print(f'  geopandas {gpd.__version__}')
print(f'  folium   {folium.__version__}')

---
## 1. Carga y descripción de datos

In [ ]:
# ─── Tabla principal Silver ────────────────────────────────────────────────────
silver = pl.read_parquet(PROC_DIR / 'silver_upz_mes.parquet')
print(f'Silver table: {silver.shape[0]:,} filas × {silver.shape[1]} columnas')
print(f'  UPZs: {silver["upz_cod"].n_unique()}')
print(f'  Años: {sorted(silver["anio"].unique().to_list())}')
print(f'  Rango n_delitos: {silver["n_delitos"].min()} – {silver["n_delitos"].max()} (media: {silver["n_delitos"].mean():.1f})')
silver.head(5)

In [ ]:
# ─── Estadísticas descriptivas ────────────────────────────────────────────────
num_cols = ['n_delitos', 'n_incidentes_nuse', 'ratio_nuse_delitos_upz',
            'temperatura_c', 'precipitacion_mm_mes',
            'cuadrantes_por_km2', 'estrato_promedio_upz',
            'n_estaciones_tm', 'dist_tm_metros']
silver.select(num_cols).describe()

In [ ]:
# ─── Nulls por columna ────────────────────────────────────────────────────────
nulls = {c: silver[c].null_count() for c in silver.columns}
nulls_df = pd.DataFrame({'columna': list(nulls.keys()),
                          'nulls': list(nulls.values()),
                          'pct': [v/len(silver)*100 for v in nulls.values()]})
nulls_df = nulls_df[nulls_df['nulls'] > 0].sort_values('pct', ascending=False)
print('Columnas con valores nulos:')
print(nulls_df.to_string(index=False))
print()
print('Nota: estrato_promedio_upz tiene ~64% nulos por cobertura parcial de F7 (44K de ~115K manzanas).')
print('Los nulos en lag features son del primer mes por UPZ (comportamiento esperado).')

In [ ]:
# ─── Datos adicionales para el EDA ────────────────────────────────────────────

# F1 DAI localidad (histórico 2018–2026)
dai = pl.read_parquet(PROC_DIR / 'delitos_localidad_anio.parquet')
print(f'DAI localidad: {dai.shape} | años: {sorted(dai["anio"].unique().to_list())}')
print(f'  Tipos de delito: {dai["tipo_delito"].unique().sort().to_list()}')

# F2 UPZ shapefile
gdf_upz = gpd.read_file(RAW_DIR / 'f2_upz.geojson')
print(f'\nUPZ shapefile: {len(gdf_upz)} polígonos | CRS: {gdf_upz.crs}')

# F6 Hurto personas PN (2003–2026, nivel municipal)
f6 = pl.read_parquet(RAW_DIR / 'f6_hurto_personas.parquet')
f6_bog = f6.filter(pl.col('municipio').str.contains('BOGOT'))\
           .with_columns(pl.col('fecha_hecho').str.slice(0,4).cast(pl.Int32).alias('anio'))
print(f'\nF6 Hurto PN Bogotá: {len(f6_bog):,} registros | años: {f6_bog["anio"].min()}–{f6_bog["anio"].max()}')

---
## 2. Visualizaciones obligatorias del concurso

Las 6 visualizaciones requeridas para la evaluación del Reto #2 Seguridad Ciudadana.

### V1 — Mapa de calor choropleta: incidentes criminales por UPZ (2025)

In [ ]:
# ─── V1: Choropleta Folium — delitos por UPZ (2025) ──────────────────────────

# Agregar delitos 2025 por UPZ
del_2025 = (
    silver.filter(pl.col('anio') == 2025)
    .group_by('upz_cod')
    .agg(pl.col('n_delitos').sum().alias('total_delitos_2025'))
    .to_pandas()
)
del_2025['upz_cod'] = del_2025['upz_cod'].astype(str)

# Merge con shapefile
gdf_del = gdf_upz.copy()
gdf_del['CODIGO_UPZ'] = gdf_del['CODIGO_UPZ'].astype(str)
gdf_del = gdf_del.merge(del_2025, left_on='CODIGO_UPZ', right_on='upz_cod', how='left')
gdf_del['total_delitos_2025'] = gdf_del['total_delitos_2025'].fillna(0)

print(f'UPZs con datos de delitos 2025: {(gdf_del["total_delitos_2025"] > 0).sum()}/{len(gdf_del)}')
print(f'Top 5 UPZs por delitos 2025:')
print(gdf_del.nlargest(5, 'total_delitos_2025')[['CODIGO_UPZ','NOMBRE','total_delitos_2025']].to_string(index=False))

# Mapa Folium choropleta
bogota_center = [4.6097, -74.0817]
m_v1 = folium.Map(location=bogota_center, zoom_start=11, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=gdf_del.__geo_interface__,
    name='Delitos por UPZ 2025',
    data=gdf_del,
    columns=['CODIGO_UPZ', 'total_delitos_2025'],
    key_on='feature.properties.CODIGO_UPZ',
    fill_color='OrRd',
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name='Total incidentes criminales NUSE 2025',
    nan_fill_color='lightgray',
).add_to(m_v1)

# Tooltip con nombre de UPZ
folium.GeoJson(
    gdf_del.__geo_interface__,
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['NOMBRE', 'CODIGO_UPZ', 'total_delitos_2025'],
        aliases=['UPZ', 'Código', 'Incidentes 2025'],
        localize=True,
    ),
).add_to(m_v1)

folium.LayerControl().add_to(m_v1)
m_v1.save(str(GRAF_DIR / 'v1_mapa_calor_upz.html'))
print('\n✓ Mapa guardado: graficas/v1_mapa_calor_upz.html')
m_v1

### V2 — Heatmap: mes × tipo de incidente criminal (2025)

In [ ]:
# ─── V2: Heatmap mes × tipo de delito ────────────────────────────────────────
# Usamos F5 raw para tener detalle por TIPO_DETALLE × MES

f5_raw = pl.read_parquet(RAW_DIR / 'f5_nuse_123.parquet')

# Filtrar 2025 y tipos criminales relevantes
TIPOS_VIZ = [
    'RIÑA', 'HURTO EFECTUADO', 'HURTO EN PROCESO',
    'MALTRATO', 'LESIONES PERSONALES',
    'NARCÓTICOS', 'DISPAROS', 'VIOLENCIA SEXUAL',
    'PORTE DE ARMAS', 'SECUESTRO'
]

hm_data = (
    f5_raw
    .filter(
        (pl.col('ANIO') == '2025') &
        (pl.col('TIPO_DETALLE').is_in(TIPOS_VIZ))
    )
    .with_columns(pl.col('MES').cast(pl.Int32, strict=False).alias('mes_int'))
    .group_by(['TIPO_DETALLE', 'mes_int'])
    .agg(pl.col('CANT_INCIDENTES').cast(pl.Int64, strict=False).sum().alias('total'))
    .to_pandas()
)

MES_NOMBRES = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
               7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}
hm_data['mes_nombre'] = hm_data['mes_int'].map(MES_NOMBRES)

# Pivot para heatmap
pivot = hm_data.pivot_table(
    values='total', index='TIPO_DETALLE', columns='mes_int', aggfunc='sum', fill_value=0
)
pivot.columns = [MES_NOMBRES.get(c, c) for c in pivot.columns]
# Ordenar por total
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    pivot, annot=True, fmt=',d', cmap='YlOrRd',
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'N° incidentes'},
)
ax.set_title('Incidentes criminales NUSE por tipo × mes — Bogotá 2025', pad=15)
ax.set_xlabel('Mes')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(GRAF_DIR / 'v2_heatmap_tipo_mes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: graficas/v2_heatmap_tipo_mes.png')

### V3 — Top 10 UPZs con más incidentes criminales (2025)

In [ ]:
# ─── V3: Top 10 UPZs por delitos 2025 ────────────────────────────────────────

top10 = (
    silver.filter(pl.col('anio') == 2025)
    .group_by('upz_cod')
    .agg(pl.col('n_delitos').sum().alias('total_delitos'))
    .sort('total_delitos', descending=True)
    .head(10)
    .to_pandas()
)

# Agregar nombre de UPZ desde shapefile
upz_nombres = gdf_upz[['CODIGO_UPZ','NOMBRE']].copy()
upz_nombres['CODIGO_UPZ'] = upz_nombres['CODIGO_UPZ'].astype(str)
top10 = top10.merge(upz_nombres, left_on='upz_cod', right_on='CODIGO_UPZ', how='left')
top10['label'] = top10.apply(
    lambda r: f"UPZ {r['upz_cod']} — {r['NOMBRE']}" if pd.notna(r['NOMBRE']) else f"UPZ {r['upz_cod']}",
    axis=1
)
top10 = top10.sort_values('total_delitos', ascending=True)  # para barh

# Colores por cuartil
colores = ['#d73027' if v >= top10['total_delitos'].quantile(0.75) else
           '#fc8d59' if v >= top10['total_delitos'].quantile(0.5) else
           '#fee090' for v in top10['total_delitos']]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top10['label'], top10['total_delitos'], color=colores)

# Etiquetas de valor
for bar, val in zip(bars, top10['total_delitos']):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

ax.set_title('Top 10 UPZs con más incidentes criminales (NUSE 2025)', pad=15)
ax.set_xlabel('Total incidentes criminales 2025')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(GRAF_DIR / 'v3_top10_upz_delitos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: graficas/v3_top10_upz_delitos.png')

### V4 — Tendencia histórica de crimen 2018–2026 (ciudad + localidades críticas)

In [ ]:
# ─── V4: Tendencia anual 2018–2026 ───────────────────────────────────────────
# Fuentes:
#   - DAI localidad (F1): totales anuales por tipo, 2018–2026, nivel localidad
#   - F6 Hurto PN Bogotá: hurto a personas municipal 2003–2026

# Tendencia ciudad total (F1 todos los tipos)
ciudad_anual = (
    dai
    .group_by('anio')
    .agg(pl.col('n_delitos').sum().alias('total_dai'))
    .sort('anio')
    .to_pandas()
)

# F6: hurto personas Bogotá (mensual → anual)
f6_anual = (
    f6_bog
    .filter((pl.col('anio') >= 2018) & (pl.col('anio') <= 2026))
    .group_by('anio')
    .agg(pl.col('cantidad').cast(pl.Int64, strict=False).sum().alias('hurto_personas'))
    .sort('anio')
    .to_pandas()
)

# Top 5 localidades por total delitos (DAI)
top5_loc = (
    dai
    .group_by('nom_localidad')
    .agg(pl.col('n_delitos').sum())
    .sort('n_delitos', descending=True)
    .head(5)['nom_localidad']
    .to_list()
)
dai_top5 = (
    dai
    .filter(pl.col('nom_localidad').is_in(top5_loc))
    .group_by(['nom_localidad', 'anio'])
    .agg(pl.col('n_delitos').sum())
    .sort(['nom_localidad', 'anio'])
    .to_pandas()
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo: DAI ciudad total + F6 hurto PN
ax1_twin = ax1.twinx()
l1, = ax1.plot(ciudad_anual['anio'], ciudad_anual['total_dai'],
               'o-', color='#e34a33', linewidth=2.5, markersize=7, label='DAI total (todas fuentes)')
l2, = ax1_twin.plot(f6_anual['anio'], f6_anual['hurto_personas'],
                    's--', color='#2166ac', linewidth=2, markersize=6, label='Hurto personas PN (F6)')
ax1.set_title('Bogotá: tendencia delitos 2018–2026', pad=12)
ax1.set_xlabel('Año')
ax1.set_ylabel('DAI — todos los delitos (localidades)', color='#e34a33')
ax1_twin.set_ylabel('Hurto personas PN (nacional)', color='#2166ac')
ax1.tick_params(axis='y', labelcolor='#e34a33')
ax1_twin.tick_params(axis='y', labelcolor='#2166ac')
lines = [l1, l2]
ax1.legend(lines, [l.get_label() for l in lines], loc='upper left', fontsize=9)
ax1.spines[['top','right']].set_visible(False)
ax1.set_xticks(sorted(ciudad_anual['anio'].unique()))

# Panel derecho: Top 5 localidades
colores_loc = ['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4']
for i, (loc, grp) in enumerate(dai_top5.groupby('nom_localidad')):
    ax2.plot(grp['anio'], grp['n_delitos'], 'o-', linewidth=2,
             color=colores_loc[i], label=loc.title(), markersize=5)

ax2.set_title('Top 5 localidades — tendencia DAI 2018–2026', pad=12)
ax2.set_xlabel('Año')
ax2.set_ylabel('Total delitos de alto impacto (DAI)')
ax2.legend(fontsize=9)
ax2.spines[['top','right']].set_visible(False)
ax2.set_xticks(sorted(dai_top5['anio'].unique()))

plt.tight_layout()
plt.savefig(GRAF_DIR / 'v4_tendencia_anual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: graficas/v4_tendencia_anual.png')
print(f'\nTop 5 localidades más críticas (DAI 2018–2026): {top5_loc}')

### V5 — Correlación lluvia vs. incidentes criminales por mes

In [ ]:
# ─── V5: Lluvia vs. delitos (scatter por mes) ─────────────────────────────────

# Agregar silver por mes (promedio ciudad)
monthly_city = (
    silver
    .group_by(['anio', 'mes'])
    .agg([
        pl.col('n_delitos').sum().alias('total_delitos_ciudad'),
        pl.col('precipitacion_mm_mes').mean().alias('lluvia_mm'),
        pl.col('temperatura_c').mean().alias('temp_c'),
    ])
    .sort(['anio', 'mes'])
    .to_pandas()
    .dropna(subset=['lluvia_mm'])
)

# Calcular correlación
corr = monthly_city['lluvia_mm'].corr(monthly_city['total_delitos_ciudad'])

# Línea de tendencia
z = np.polyfit(monthly_city['lluvia_mm'], monthly_city['total_delitos_ciudad'], 1)
p = np.poly1d(z)
x_line = np.linspace(monthly_city['lluvia_mm'].min(), monthly_city['lluvia_mm'].max(), 100)

# Colorear por mes (estacionalidad)
mes_colors = plt.cm.RdYlBu_r(np.linspace(0, 1, 12))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Scatter lluvia vs. delitos
scatter = ax1.scatter(
    monthly_city['lluvia_mm'],
    monthly_city['total_delitos_ciudad'],
    c=monthly_city['mes'], cmap='RdYlBu_r',
    s=80, alpha=0.8, edgecolors='white', linewidths=0.5,
)
ax1.plot(x_line, p(x_line), 'k--', linewidth=1.5, alpha=0.7, label=f'Tendencia (r={corr:.3f})')
plt.colorbar(scatter, ax=ax1, label='Mes')
ax1.set_title(f'Precipitación vs. incidentes criminales\n(r = {corr:.3f})', pad=12)
ax1.set_xlabel('Precipitación mensual acumulada (mm)')
ax1.set_ylabel('Total incidentes criminales ciudad')
ax1.legend()
ax1.spines[['top','right']].set_visible(False)

# Boxplot por mes (estacionalidad)
monthly_city['mes_nombre'] = monthly_city['mes'].map(
    {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
     7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}
)
orden_meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
valid_meses = [m for m in orden_meses if m in monthly_city['mes_nombre'].values]

ax2.plot(monthly_city.sort_values('mes')['mes_nombre'].unique() if len(monthly_city) > 1 else valid_meses,
         [monthly_city[monthly_city['mes_nombre']==m]['total_delitos_ciudad'].mean() for m in valid_meses],
         'o-', color='#e34a33', linewidth=2, markersize=7, label='Media incidentes')
ax2_twin = ax2.twinx()
ax2_twin.bar(valid_meses,
             [monthly_city[monthly_city['mes_nombre']==m]['lluvia_mm'].mean() for m in valid_meses],
             alpha=0.4, color='#2166ac', label='Lluvia media (mm)')
ax2.set_title('Estacionalidad mensual: delitos vs. lluvia', pad=12)
ax2.set_xlabel('Mes')
ax2.set_ylabel('Total incidentes (media)', color='#e34a33')
ax2_twin.set_ylabel('Precipitación media (mm)', color='#2166ac')
ax2.tick_params(axis='y', labelcolor='#e34a33')
ax2_twin.tick_params(axis='y', labelcolor='#2166ac')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(GRAF_DIR / 'v5_lluvia_vs_delitos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Guardado: graficas/v5_lluvia_vs_delitos.png')
print(f'\nCorrelación Pearson (lluvia × delitos): r = {corr:.4f}')
print(f'  → {"Correlación negativa débil" if corr < -0.1 else "Sin correlación significativa" if abs(corr) < 0.1 else "Correlación positiva"} entre lluvia y crimen.')

### V6 — Distribución de delitos por nivel de estrato promedio (UPZ)

In [ ]:
# ─── V6: Distribución delitos por estrato UPZ ─────────────────────────────────
# Usa los 43 UPZs con datos de estrato (F7 parcial)
# Este análisis es fundamental para el análisis de sesgo del modelo (Notebook 04)

silver_estrato = (
    silver
    .filter(pl.col('estrato_promedio_upz').is_not_null())
    .with_columns(
        pl.when(pl.col('estrato_promedio_upz') < 1.5).then(pl.lit('Bajo (< 1.5)'))
         .when(pl.col('estrato_promedio_upz') < 2.5).then(pl.lit('Medio-bajo (1.5–2.5)'))
         .when(pl.col('estrato_promedio_upz') < 3.5).then(pl.lit('Medio (2.5–3.5)'))
         .otherwise(pl.lit('Medio-alto (> 3.5)'))
         .alias('grupo_estrato')
    )
    .to_pandas()
)

print(f'Registros con estrato: {len(silver_estrato):,} de {len(silver):,} ({100*len(silver_estrato)/len(silver):.1f}%)')
print(f'Distribución por grupo estrato:')
print(silver_estrato.groupby('grupo_estrato')['upz_cod'].nunique().sort_index())

# Determinar orden de grupos presente en los datos
orden_estrato = ['Bajo (< 1.5)', 'Medio-bajo (1.5–2.5)', 'Medio (2.5–3.5)', 'Medio-alto (> 3.5)']
grupos_presentes = [g for g in orden_estrato if g in silver_estrato['grupo_estrato'].values]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot n_delitos por grupo estrato
colores_estrato = {'Bajo (< 1.5)': '#d73027',
                   'Medio-bajo (1.5–2.5)': '#fc8d59',
                   'Medio (2.5–3.5)': '#fee090',
                   'Medio-alto (> 3.5)': '#91bfdb'}
palette = {g: colores_estrato[g] for g in grupos_presentes}

sns.boxplot(
    data=silver_estrato,
    x='grupo_estrato', y='n_delitos',
    order=grupos_presentes,
    palette=palette, ax=ax1,
    showfliers=True,
)
ax1.set_title('Distribución de incidentes criminales\npor estrato socioeconómico (UPZ)', pad=12)
ax1.set_xlabel('Grupo de estrato promedio UPZ')
ax1.set_ylabel('Incidentes criminales por mes')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=15, ha='right')
ax1.spines[['top','right']].set_visible(False)

# Scatter estrato_promedio vs. media de delitos por UPZ
upz_estrato_media = (
    silver_estrato.groupby('upz_cod')
    .agg(media_delitos=('n_delitos', 'mean'),
         estrato=('estrato_promedio_upz', 'mean'))
    .reset_index()
)

scatter = ax2.scatter(
    upz_estrato_media['estrato'],
    upz_estrato_media['media_delitos'],
    c=upz_estrato_media['estrato'], cmap='RdYlBu_r',
    s=100, alpha=0.8, edgecolors='white',
)

# Línea de tendencia
if len(upz_estrato_media) > 2:
    z = np.polyfit(upz_estrato_media['estrato'], upz_estrato_media['media_delitos'], 1)
    p = np.poly1d(z)
    x_r = np.linspace(upz_estrato_media['estrato'].min(), upz_estrato_media['estrato'].max(), 50)
    corr_est = upz_estrato_media['estrato'].corr(upz_estrato_media['media_delitos'])
    ax2.plot(x_r, p(x_r), 'k--', linewidth=1.5, label=f'r = {corr_est:.2f}')
    ax2.legend()

plt.colorbar(scatter, ax=ax2, label='Estrato promedio UPZ')
ax2.set_title('Estrato promedio UPZ vs.\nmedia de incidentes criminales/mes', pad=12)
ax2.set_xlabel('Estrato promedio UPZ')
ax2.set_ylabel('Media incidentes criminales/mes')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(GRAF_DIR / 'v6_estrato_vs_delitos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: graficas/v6_estrato_vs_delitos.png')
print()
print('⚠ NOTA SOBRE SESGO: Esta visualización usa solo 43 UPZs con datos de estrato.')
print('  Los datos disponibles provienen principalmente de estratos bajos (0–3).')
print('  El análisis completo de sesgo del modelo se realizará en Notebook 04.')

---
## 3. Análisis complementario

In [ ]:
# ─── Distribución de n_delitos (asimetría — desbalance de clases para el modelo) ──

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histograma n_delitos
ax1.hist(silver['n_delitos'].to_numpy(), bins=50, color='#e34a33', alpha=0.8, edgecolor='white')
ax1.axvline(silver['n_delitos'].mean(), color='black', linestyle='--', linewidth=2,
            label=f'Media: {silver["n_delitos"].mean():.0f}')
ax1.axvline(silver['n_delitos'].quantile(0.75), color='navy', linestyle=':', linewidth=2,
            label=f'P75: {silver["n_delitos"].quantile(0.75):.0f}')
ax1.set_title('Distribución de incidentes criminales por UPZ/mes\n(muestra el desbalance de clases para el modelo)')
ax1.set_xlabel('Incidentes criminales/mes')
ax1.set_ylabel('Frecuencia')
ax1.legend()
ax1.spines[['top','right']].set_visible(False)

# Cobertura policial vs. delitos
silver_cuad = silver.filter(pl.col('cuadrantes_por_km2').is_not_null()).to_pandas()
if len(silver_cuad) > 10:
    upz_cuad = silver_cuad.groupby('upz_cod').agg(
        media_delitos=('n_delitos', 'mean'),
        cuad_km2=('cuadrantes_por_km2', 'mean')
    ).reset_index()
    ax2.scatter(upz_cuad['cuad_km2'], upz_cuad['media_delitos'],
                alpha=0.6, s=60, color='#2166ac', edgecolors='white')
    z = np.polyfit(upz_cuad['cuad_km2'], upz_cuad['media_delitos'], 1)
    p = np.poly1d(z)
    x_r = np.linspace(upz_cuad['cuad_km2'].min(), upz_cuad['cuad_km2'].max(), 50)
    corr_c = upz_cuad['cuad_km2'].corr(upz_cuad['media_delitos'])
    ax2.plot(x_r, p(x_r), 'k--', linewidth=1.5, label=f'r = {corr_c:.2f}')
    ax2.set_title('Cobertura policial vs. incidentes criminales por UPZ')
    ax2.set_xlabel('Cuadrantes por km²')
    ax2.set_ylabel('Media incidentes criminales/mes')
    ax2.legend()
    ax2.spines[['top','right']].set_visible(False)
    print(f'Correlación cuadrantes/km² vs. media delitos: r = {corr_c:.3f}')

plt.tight_layout()
plt.savefig(GRAF_DIR / 'v7_distribucion_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: graficas/v7_distribucion_cobertura.png')

In [ ]:
# ─── Resumen de la tabla Silver ───────────────────────────────────────────────

print('=' * 60)
print('RESUMEN TABLA SILVER — silver_upz_mes.parquet')
print('=' * 60)
print(f'  Filas:     {silver.shape[0]:,}')
print(f'  Columnas:  {silver.shape[1]}')
print(f'  UPZs:      {silver["upz_cod"].n_unique()}')
print(f'  Años:      {silver["anio"].min()} – {silver["anio"].max()}')
print(f'  n_delitos: {silver["n_delitos"].min()} – {silver["n_delitos"].max()} (media {silver["n_delitos"].mean():.1f})')
print()
print('Columnas disponibles para Gold (Notebook 03):')
for c in silver.columns:
    null_pct = silver[c].null_count() / len(silver) * 100
    status = f' ⚠ {null_pct:.0f}% null' if null_pct > 5 else ' ✓'
    print(f'  {c:<35}{status}')
print()
print('LIMITACIONES CONOCIDAS:')
print('  1. Datos de crimen (F5 NUSE) solo disponibles 2025–2026')
print('     → El modelo se entrenará con 2025 y se evaluará en 2026')
print('  2. Estrato (F7): solo 43/120 UPZs cubiertas')
print('     → Imputación por vecinos más cercanos en Gold (Notebook 03)')
print('  3. F1 DAI solo disponible a nivel localidad (no UPZ)')
print('     → Usada como referencia histórica, no como variable del modelo')

---
## 4. Qué sigue → Gold (Notebook 03)

El análisis exploratorio confirma que la tabla Silver tiene las variables candidatas para el modelo XGBoost.  
En el **Notebook 03 (Features / Gold)** se realizará:

1. **Definición de `nivel_riesgo`** (Y):  
   - ALTO: UPZs en percentil 75+ de n_delitos por mes  
   - MEDIO: percentil 25–75  
   - BAJO: percentil < 25  

2. **Ingeniería de features**:  
   - Correlación entre variables (eliminar colineales, VIF)  
   - Imputación de nulos en `estrato_promedio_upz`  
   - Encoding de `tipo_delito_dominante`  
   - Adición de features temporales: `dia_semana`, `es_fin_semana`, `franja_horaria`  

3. **Selección final de las 14 variables** del modelo  
   - Ver `PROYECTO.md` para la lista definitiva

4. **Guardar `tabla_maestra_upz.parquet`** → input para Notebook 04 (XGBoost)

---

### Hallazgos clave del EDA

| Hallazgo | Implicación para el modelo |
|----------|---------------------------|
| Distribución muy sesgada de n_delitos (cola derecha) | Esperar desbalance de clases ALTO/BAJO; usar `scale_pos_weight` en XGBoost |
| Correlación lluvia–crimen débil o negativa | La lluvia es una feature secundaria; incluir pero no sobreponderar |
| 43/120 UPZs con estrato disponible | Imputar por KNN antes del modelo |
| F5 NUSE solo 2025–2026 | Split temporal: TRAIN = Jan-Oct 2025, TEST = Nov 2025-2026 |
| RIÑA domina en incidentes (599K) | Tipo de delito dominante = feature relevante |
| F4 tiene nombre de CAI (`PCUNOMCAI`) | **No se necesita cai_bogota.csv manual** — Módulo 3 usa F4 directamente |